# 2.LCEL文档填充链

知识点讲解：Stuff 文档填充链（create_stuff_documents_chain）

1. 什么是文档填充链（Stuff Documents Chain）
   - 将多个 Document 对象的内容拼接后，"塞入"（stuff）提示词的上下文变量中
   - 是 RAG 检索增强生成流程中最简单直接的文档处理策略
   - 适用于文档总量较小、能完整放入模型上下文窗口的场景

2. Document 对象的结构
   - page_content: str - 文档正文内容，是拼接的主要数据源
   - metadata: dict - 文档元数据（来源、页码、时间等），默认不参与拼接
   - 通常由 DocumentLoader 加载或由 Retriever 检索产出

3. create_stuff_documents_chain 的工作原理
   - 接收一个 llm 和一个 prompt，返回一个 Runnable 链
   - 内部用 format_document 将每个 Document 格式化为字符串
   - 用 document_separator（默认 "

"）连接所有文档内容
   - 将拼接结果注入 prompt 的 context 变量（由 document_variable_name 指定，默认 "context"）

4. 四种常见的文档处理策略对比
   - Stuff：全部塞入、一次调用，最简单但受上下文窗口限制
   - MapReduce：先对每篇文档单独处理再汇总，可处理大量文档
   - Refine：迭代式逐个文档精炼答案，质量高但耗时长
   - MapRerank：对每篇文档独立打分，选取最佳答案

5. 提示词中 context 变量的约定
   - prompt 必须包含 {context} 占位符，否则会抛出异常
   - 推荐用 XML 标签（如 <context>...</context>）包裹，帮助模型区分上下文边界
   - 其他变量（如 {query}）由调用方在 invoke 时传入

6. 典型输出示例与观察点（课程示例）
   - 对「大家都喜欢什么颜色」提问，传入三篇文档，模型会跨文档归纳：
       根据提供的信息：
       - 小明喜欢绿色，不喜欢黄色。
       - 小王喜欢粉色，也有一点喜欢红色。
       - 小泽喜欢蓝色，但更喜欢青色。
       所以，小明喜欢绿色，小王喜欢粉色和红色（少许），小泽喜欢青色和蓝色。
   - 观察点：Stuff 策略把全部上下文一次性喂给模型，由其自行跨文档汇总

7. 注意事项与最佳实践
   - 文档过多会超出上下文窗口，导致报错或内容截断
   - 生产环境建议先用 Retriever 筛选 Top-K 相关文档再填充
   - 可通过 document_prompt 自定义单个文档的格式（如包含来源信息）

8. 数据流转过程
   - {"query": str, "context": list[Document]}
   - → 文档拼接：list[Document] → str（以 document_separator 连接）
   - → prompt 填充：{"query": str, "context": str} → Messages
   - → llm 推理：Messages → AIMessage → 输出解析 → str


In [ ]:
import dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个强大的聊天机器人，能根据用户提供的上下文来回复用户的问题。\n\n<context>{context}</context>"),
    ("human", "{query}")
])
llm = ChatOpenAI(model="deepseek-v4-pro")
chain = create_stuff_documents_chain(prompt=prompt, llm=llm)
documents = [
    Document(page_content="小明喜欢绿色，但不喜欢黄色"),
    Document(page_content="小王喜欢粉色，也有一点喜欢红色"),
    Document(page_content="小泽喜欢蓝色，但更喜欢青色"),
]
content = chain.invoke({"query": "请帮我统计一下大家都喜欢什么颜色", "context": documents})
print(content)